# Fault Tolerance

Kafi Streams ensures fault tolerance with checkpointing.

We explain how it works in [Checkpointing](#checkpointing).

The way how fault tolerance is implemented in a stream processor is a main determining factor for its [delivery guarantees](#guarantees).

Finally, we show checkpointing in action by a practical [example](#example).

Notice that fault tolerance is only supported by *Streams*, not the *TopologyNode* class as only the former includes support for persistence (through Kafi).


## Overview

[Preparation](#prep)

* [Checkpointing](#checkpointing)
  * [Enabling checkpointing](#enabling)
  * [Checkpointing in detail](#detail)
* [Delivery guarantees](#guarantees)
* [Example](#example)


---
<a id="prep"></a>
## Preparation

Before we start off, we first prepare for the examples to follow:

In [17]:
import sys
sys.path.insert(1, "../..")

from kafi.streams.streams import Streams

from generators import OrderGenerator
order_generator = OrderGenerator()

order_source_str = "orders"
sink_str = "sink"


---
<a id="checkpointing"></a>
## Checkpointing

How does checkpointing work in Kafi Streams?

This global state can be stored in any *storage* supported by Kafi, i.e., currently, Kafka itself, or, via Kafi's *emulated Kafka*, to disk, S3 or Azure Blob Storage.

[Enabling checkpointing](#enabling) explains how to enable checkpointing, and [Checkpointing in detail](#detail) describes how checkpointing is implemented in detail and embedded into the consume + process + produce loop of the *Streams* class.


<a id="enabling"></a>
### Enabling checkpointing

Let's revisit the signature of the `start_streams` method of the *Streams* class to see how checkpointing can be enabled:

```
def start_streams(built_tn, checkpoint_storage=None, checkpoint_topic_str=None, checkpoint_interval_float=default_checkpoint_interval_float, **kwargs):
    """Run streams() in a background thread; returns a function to stop it.

    Args:
        built_tn: built tn to run
        checkpoint_storage: storage backend for checkpoints, or None to disable checkpointing
        checkpoint_topic_str: topic name used to store checkpoints
        checkpoint_interval_float: seconds between checkpoints
        **kwargs: passed through to streams()
    Returns:
        stop_fun: None -> None function to stop the Streams processing thread"""

```

You can enable checkpointing by setting `checkpoint_storage` and `checkpoint_topic_str` to the Kafi stroage and topic to be used for the checkpointing.

The `checkpoint_interval_float` is a floating point number specifying the checkpoint interval (in seconds).


<a id="detail"></a>
### Checkpointing in detail

*Streams* implements the typical consume + process + produce loop from stream processing. Let's look at it in more detail in the case if checkpointing is enabled:

* before the loop: read the last checkpoint if there is any and set the global state and the offsets of the consumer group for the source topics accordingly,
* in the loop:
  1. consume new data from the source topics,
  2. process the new data and return the new resulting data,
  3. produce the new resulting data to the sink topics.
  4. if there is new resulting data and the checkpoint interval is exceeded:  
     4.1 save the checkpoint = the current global state + the offsets of the last processed messages from the source topics,  
     4.2 commit these offsets to Kafka.


---
<a id="guarantees"></a>
## Delivery guarantees

In the previous sub-section, you have seen how checkpointing works in detail in Kafi Streams.

Contrary to classical stream processors like Kafka Streams and Flink, Kafi Streams does not use individual state stores per operator but one global state, i.e., the state of the underlying pydbsp circuit.

Hence, saving a checkpoint to Kafka (or disk, S3 or Azure Blob Storage using Kafi's *emulated Kafka*) takes time.

Now consider how exactly-once semantics would have to be implemented with Kafka transactions:

* before the loop: read the last checkpoint if there is any and set the global state and the offsets of the consumer group for the source topics accordingly,
* in the loop:
  1. consume new data from the source topics,
  2. process the new data and return the new resulting data,
  3. ( start the Kafka transaction,  
     3.1. produce the new resulting data to the sink topics.  
     3.2. save the checkpoint = the current global state + the offsets of the last processed messages from the source topics,  
     3.3. commit the Kafka transaction )

What does this mean? To be able to use Kafka's transactions, we would have save the checkpoints in *every* step of our consume + process + consume loop. Infeasible.

Hence, to keep it simple, Kafi Streams only supports *at-least once* delivering guarantees.


---
<a id="example"></a>
## Example

This section shows an example of checkpointing in Kafi Streams.

We start with setting up a [topology](#topology)

<a id="topology"></a>
### Topology

The topology has one source (orders) and aggregates these orders per `customer_id`:
* `orders` the number of orders of the customer,
* `order_ids` the `order_id`s of the customer,
* `total_price` the sum of the `price`s of the orders of the customer:


In [40]:
import sys
sys.path.insert(1, "../..")

import kafi.streams.streams
import importlib
importlib.reload(kafi.streams.streams)

from kafi.kafka.cluster.cluster import Cluster
from kafi.streams.streams import Streams

import logging
logging.basicConfig(level=logging.DEBUG)

c = Cluster({"kafka": {"bootstrap.servers": "localhost:9092"}})

source_str = "orders"
sink_str = "orders_aggregated"

tn = (
    Streams.source(c, source_str)

    .map(lambda r: r["value"])
    .group_by_agg(lambda r: r["customer_id"],
                  lambda r: r,
                  lambda agg_r, r: {"orders": agg_r["orders"] + 1,
                                    "order_ids": sorted(agg_r["order_ids"] + [r["order_id"]]),
                                    "total_price": agg_r["total_price"] + r["price"]},
                  {"orders": 0, "order_ids": [], "total_price": 0},
                  lambda by, agg_r: {"customer_id": by,
                                     "orders": agg_r["orders"],
                                     "order_ids": agg_r["order_ids"],
                                     "total_price": agg_r["total_price"]})
    .map(lambda r: {"key": r["customer_id"],
                    "value": r}).peek("sink")
    .sink(c, sink_str)
)

built_tn = Streams.build(tn)



<a id="step_1"></a>
### Step 1

In step 1, we:
* start the *Streams* thread using the `start_streams()` method (with checkpointing enabled),
* generate `1000` orders, produce them to the source topic and let *Streams* thread process them:

In [ ]:
from kafi.helpers import get_millis

orders_int = 1000

built_tn.reset()

checkpoint_str = "checkpoint"
g = f"group_{get_millis()}"

c.recreate(source_str)
c.recreate(sink_str)
c.recreate(checkpoint_str)

#

stop_fun = Streams.start_streams(built_tn, checkpoint_storage=c, checkpoint_topic_str=checkpoint_scheckpoint_interval_floatrval=0.01, group=g)

#

pr = c.producer(source_str)
gen = OrderGenerator()

#

for _ in range(orders_int):
    m = gen.generate()
    pr.produce_list([m])

#

pr.close()



INFO:kafi.streams.streams:Starting Streams...
DEBUG:kafi.streams.streams:Checkpoint consumer group ('group_1787431634344_checkpoint') offsets for topic 'checkpoint': {}


(['checkpoint'], 'group_1787431634344_checkpoint')


Consuming: 0 msg [00:00, ? msg/s]

'orders'

DEBUG:kafi.streams.streams:Source consumer group ('group_1787431634344') offsets for topic 'orders': {}



(['orders'], 'group_1787431634344')


INFO:kafi.streams.streams:Saving checkpoint...
INFO:kafi.streams.streams:...saving checkpoint done (7 KB compressed, 20 KB uncompressed).
INFO:kafi.streams.streams:Committed {0: 10} for source orders.


sink: {'key': 2, 'value': {'customer_id': 2, 'orders': 2, 'order_ids': ['0bdbcb6a-5eb5-44b9-8149-4b8aa823545e', '6f0baf82-a650-4c6c-97cf-3b5c08de9735'], 'total_price': 119.19999999999999}}
sink: {'key': 0, 'value': {'customer_id': 0, 'orders': 1, 'order_ids': ['fd3c56a4-db25-4c10-b381-abc8ebd9b0da'], 'total_price': 31.0}}
sink: {'key': 1, 'value': {'customer_id': 1, 'orders': 1, 'order_ids': ['d6274dc7-8ef2-435c-8a58-167cfc8937e7'], 'total_price': 89.54}}
sink: {'key': 7, 'value': {'customer_id': 7, 'orders': 3, 'order_ids': ['0ede1e95-f7b3-4e5a-8e9d-55d47c273edc', '214b1dbf-7858-4f42-b73e-2766f3feda80', 'd7ea34b2-954c-4c26-9cbe-12c31299d91c'], 'total_price': 132.43}}
sink: {'key': 6, 'value': {'customer_id': 6, 'orders': 2, 'order_ids': ['32db3c6a-fff9-47ce-b8ed-5ce01de1ca51', '999aa6e4-f159-4eb8-9390-446ba9202560'], 'total_price': 91.27}}
sink: {'key': 5, 'value': {'customer_id': 5, 'orders': 1, 'order_ids': ['4a4e0624-419b-4b3b-8910-696498b9f328'], 'total_price': 1.09}}


As you see in the output, a first checkpoint has been saved to the checkpoint topic.

<a id="step_2"></a>
### Step 2

In step 2, we stop the *Streams* thread:

In [29]:
stop_fun()
Streams.threads()

INFO:kafi.streams.streams:Safely stopping Streams...
INFO:kafi.streams.streams:...done.


[]

<a id="step_3"></a>
### Step 3

In step 3, we:
* reset the state of the built topology node (`built_tn`),
* (re-)start the *Streams* thread using the `start_streams()` method (with checkpointing enabled),
* generate another `1000` orders, produce them to the source topic and let *Streams* thread process them:

In [ ]:
built_tn.reset()

print(c.l(source_str))
print(c.l(sink_str))
print(c.l(checkpoint_str))

#

stop_fun = Streams.start_streams(built_tn, checkpoint_storage=c, checkpoint_topic_str=checkpoint_str, group=g)

#

pr = c.producer(source_str)

#

for _ in range(orders_int):
    m = gen.generate()
    pr.produce_list([m])

#

pr.close()



INFO:kafi.streams.streams:Starting Streams...


{'orders': 10}
{'orders_aggregated': 6}
{'checkpoint': 8}


'orders'

DEBUG:kafi.streams.streams:Checkpoint consumer group ('group_1787431634344_checkpoint') offsets for topic 'checkpoint': {}


(['checkpoint'], 'group_1787431634344_checkpoint')


Consuming: 0 msg [00:00, ? msg/s]

INFO:kafi.streams.streams:Loading checkpoint...
INFO:kafi.streams.streams:...loading checkpoint done (7 KB compressed, 20 KB uncompressed).
DEBUG:kafi.streams.streams:Source consumer group ('group_1787431634344') offsets for topic 'orders': {0: 10}
DEBUG:kafi.streams.streams:Source consumer group offsets for topic 'orders' overridden by checkpoint offsets: {0: 10}



(['checkpoint'], 'group_1787431634344_checkpoint')
(['orders'], 'group_1787431634344')


INFO:kafi.streams.streams:Saving checkpoint...
INFO:kafi.streams.streams:...saving checkpoint done (8 KB compressed, 27 KB uncompressed).
INFO:kafi.streams.streams:Committed {0: 20} for source orders.


sink: {'key': 2, 'value': {'customer_id': 2, 'orders': 4, 'order_ids': ['0bdbcb6a-5eb5-44b9-8149-4b8aa823545e', '6f0baf82-a650-4c6c-97cf-3b5c08de9735', '8108a37e-0483-4254-8f73-5282e34ac2e2', 'e43f7c31-15ca-4443-92ab-8ece321ec853'], 'total_price': 189.79999999999998}}
sink: {'key': 5, 'value': {'customer_id': 5, 'orders': 3, 'order_ids': ['43ba7c31-d378-488c-bb4d-4ec4386a50fa', '4a4e0624-419b-4b3b-8910-696498b9f328', '4f496cc6-d37d-43fb-b9dc-5bd23c54f018'], 'total_price': 131.84}}
sink: {'key': 7, 'value': {'customer_id': 7, 'orders': 5, 'order_ids': ['0ede1e95-f7b3-4e5a-8e9d-55d47c273edc', '1cf558b3-8900-4567-8167-d9e1f3920def', '214b1dbf-7858-4f42-b73e-2766f3feda80', '761587e0-404e-4d58-a157-6288917dfb3b', 'd7ea34b2-954c-4c26-9cbe-12c31299d91c'], 'total_price': 244.48}}
sink: {'key': 4, 'value': {'customer_id': 4, 'orders': 1, 'order_ids': ['1d93a4d5-43c8-4e63-83ad-aa55b4a316f1'], 'total_price': 18.4}}
sink: {'key': 9, 'value': {'customer_id': 9, 'orders': 2, 'order_ids': ['4b46efc0-

In the output log, you can see that the checkpoint from the previous processing ([step 1](#step_1)) is correctly loaded and thus the state recovered.

<a id="step_4"></a>
### Step 4

In step 4, we stop the *Streams* thread once again:

In [31]:
stop_fun()
Streams.threads()

INFO:kafi.streams.streams:Safely stopping Streams...
INFO:kafi.streams.streams:...done.


[]

<a id="step_5"></a>
### Step 5

In step 5, we again:
* reset the state of the built topology node (`built_tn`),
* (re-)start the *Streams* thread using the `start_streams()` method (with checkpointing enabled),
* generate another `1000` orders, produce them to the source topic and let *Streams* thread process them:

In [ ]:
built_tn.reset()

print(c.l(source_str))
print(c.l(sink_str))
print(c.l(checkpoint_str))

#

stop_fun = Streams.start_streams(built_tn, checkpoint_storage=c, checkpoint_topic_str=checkpoint_str, group=g)

#

pr = c.producer(source_str)

#

for _ in range(orders_int):
    m = gen.generate()
    pr.produce_list([m])

#

pr.close()


INFO:kafi.streams.streams:Starting Streams...


{'orders': 20}
{'orders_aggregated': 12}
{'checkpoint': 18}


'orders'

DEBUG:kafi.streams.streams:Checkpoint consumer group ('group_1787431634344_checkpoint') offsets for topic 'checkpoint': {0: 8}


(['checkpoint'], 'group_1787431634344_checkpoint')


Consuming: 0 msg [00:00, ? msg/s]

INFO:kafi.streams.streams:Loading checkpoint...
INFO:kafi.streams.streams:...loading checkpoint done (8 KB compressed, 27 KB uncompressed).
DEBUG:kafi.streams.streams:Source consumer group ('group_1787431634344') offsets for topic 'orders': {0: 20}
DEBUG:kafi.streams.streams:Source consumer group offsets for topic 'orders' overridden by checkpoint offsets: {0: 20}



(['checkpoint'], 'group_1787431634344_checkpoint')
(['orders'], 'group_1787431634344')


INFO:kafi.streams.streams:Saving checkpoint...
INFO:kafi.streams.streams:...saving checkpoint done (9 KB compressed, 30 KB uncompressed).
INFO:kafi.streams.streams:Committed {0: 30} for source orders.


sink: {'key': 5, 'value': {'customer_id': 5, 'orders': 5, 'order_ids': ['43ba7c31-d378-488c-bb4d-4ec4386a50fa', '4a4e0624-419b-4b3b-8910-696498b9f328', '4f496cc6-d37d-43fb-b9dc-5bd23c54f018', 'ad74fa9f-90de-4c69-9106-004f3959a1f8', 'f37ddb6a-82b6-44c6-8590-ceb3748c9fd6'], 'total_price': 156.77}}
sink: {'key': 2, 'value': {'customer_id': 2, 'orders': 5, 'order_ids': ['0bdbcb6a-5eb5-44b9-8149-4b8aa823545e', '6f0baf82-a650-4c6c-97cf-3b5c08de9735', '8108a37e-0483-4254-8f73-5282e34ac2e2', '8c328f70-11a4-48fb-9fce-2aeaa9acc2ce', 'e43f7c31-15ca-4443-92ab-8ece321ec853'], 'total_price': 259.34}}
sink: {'key': 9, 'value': {'customer_id': 9, 'orders': 4, 'order_ids': ['4b46efc0-eee0-4afb-93e1-cc9a8713b26c', 'c9c9a27e-5a4b-423e-a282-417738d6f16c', 'd3042cd3-c43b-4e45-bf4d-f8518c802dfb', 'fa1fc024-ddaf-43e9-a826-7d417b7edd4a'], 'total_price': 145.20999999999998}}
sink: {'key': 1, 'value': {'customer_id': 1, 'orders': 2, 'order_ids': ['4a807a09-e256-4093-b8c4-3186be5d2a10', 'd6274dc7-8ef2-435c-8a58-

<a id="step_6"></a>
### Step 6

In step 6, we stop the *Streams* thread a last time:


In [33]:
stop_fun()
Streams.threads()

INFO:kafi.streams.streams:Safely stopping Streams...
INFO:kafi.streams.streams:...done.


[]

<a id="step_7"></a>
### Step 7

In the last step 7, we:
* read both the source and the sink topic,
* calculate the aggregations outside of Kafi Streams,
* and compare them to the aggregations read from the sink topic:

In [39]:
import math

source_key_int_value_dict_dict = {}
source_m_list = c.cat(source_str)
for m in source_m_list:
    customer_id_int = m["value"]["customer_id"]
    order_id_str = m["value"]["order_id"]
    price_int = m["value"]["price"]
    #
    agg_orders_int = source_key_int_value_dict_dict.get(customer_id_int, {}).get("orders", 0)
    agg_order_id_str_list = source_key_int_value_dict_dict.get(customer_id_int, {}).get("order_ids", [])
    agg_total_price_int = source_key_int_value_dict_dict.get(customer_id_int, {}).get("total_price", 0)
    #
    source_key_int_value_dict_dict[customer_id_int] = {"customer_id": customer_id_int,
                                                       "orders": agg_orders_int + 1,
                                                       "order_ids": sorted(agg_order_id_str_list + [order_id_str]),
                                                       "total_price": agg_total_price_int + price_int}

#

sink_key_int_value_dict_dict = {}
sink_m_list = c.cat(sink_str)
for m in sink_m_list:
    sink_key_int_value_dict_dict[m["value"]["customer_id"]] = m["value"]

#

print(source_key_int_value_dict_dict)
print(sink_key_int_value_dict_dict)

for key_int, value_dict in source_key_int_value_dict_dict.items():
    keys_match_bool = value_dict["customer_id"] == sink_key_int_value_dict_dict[key_int]["customer_id"]
    orders_match_bool = value_dict["orders"] == sink_key_int_value_dict_dict[key_int]["orders"]
    order_ids_match_bool = value_dict["order_ids"] == sink_key_int_value_dict_dict[key_int]["order_ids"]
    price_match_bool = math.isclose(
        value_dict["total_price"], sink_key_int_value_dict_dict[key_int]["total_price"], rel_tol=1e-9)
    #
    if not (keys_match_bool and orders_match_bool and order_ids_match_bool and price_match_bool):
        print("First mismatch:")
        print("Source:", source_dict)
        print("Sink:  ", sink_dict)
        raise Exception("Test failed")
#

print("Test successful.")


(['orders'], '1787432788594')


Consuming: 0 msg [00:00, ? msg/s]


(['orders_aggregated'], '1787432793819')


Consuming: 0 msg [00:00, ? msg/s]


{2: {'customer_id': 2, 'orders': 5, 'order_ids': ['0bdbcb6a-5eb5-44b9-8149-4b8aa823545e', '6f0baf82-a650-4c6c-97cf-3b5c08de9735', '8108a37e-0483-4254-8f73-5282e34ac2e2', '8c328f70-11a4-48fb-9fce-2aeaa9acc2ce', 'e43f7c31-15ca-4443-92ab-8ece321ec853'], 'total_price': 259.34}, 0: {'customer_id': 0, 'orders': 4, 'order_ids': ['157fd799-7244-406f-9854-1f151123184f', '1b393399-3e5f-434c-87f0-4c8d3a85d3ea', '79f8c0dd-80d1-401f-a1da-f6de4bd7391f', 'fd3c56a4-db25-4c10-b381-abc8ebd9b0da'], 'total_price': 109.78}, 1: {'customer_id': 1, 'orders': 2, 'order_ids': ['4a807a09-e256-4093-b8c4-3186be5d2a10', 'd6274dc7-8ef2-435c-8a58-167cfc8937e7'], 'total_price': 111.94}, 7: {'customer_id': 7, 'orders': 5, 'order_ids': ['0ede1e95-f7b3-4e5a-8e9d-55d47c273edc', '1cf558b3-8900-4567-8167-d9e1f3920def', '214b1dbf-7858-4f42-b73e-2766f3feda80', '761587e0-404e-4d58-a157-6288917dfb3b', 'd7ea34b2-954c-4c26-9cbe-12c31299d91c'], 'total_price': 244.48000000000002}, 6: {'customer_id': 6, 'orders': 2, 'order_ids': ['

Because in this step, we read the entire source topic which has been subsequently filled with new data after we had intentionally stopped the *Streams* thread, and the entire sink topic, and because all the source data is randomly generated, this steps show that the checkpointing in Kafi Streams works :)
